# **Portfolio:** Mario Casanova — Data Science & Analytics Portfolio
## **Case Study:** Algorithmic Asset Allocation (Hierarchical Risk Parity)

---
*Nota Institucional: La disrupción schumpeteriana en la teoría de asignación de capital puede codificarse mediante un optimizador de carteras que trascienda la varianza media de Markowitz. Este notebook implementa Hierarchical Risk Parity (HRP), aplicando teoría de grafos y métodos de agrupamiento jerárquico no supervisado para particionar activos según su topología de correlación. La arquitectura resultante elimina la inestabilidad algorítmica inherente a la inversión de matrices de covarianza tradicionales.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import linkage, dendrogram, leaves_list
from scipy.spatial.distance import squareform
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
sns.set_palette("mako")
np.random.seed(42)

In [ ]:
# Load deterministic dataset
df = pd.read_csv('../data/correlated_assets_synthetic.csv', index_col=0, parse_dates=True)
returns = df.pct_change().dropna()

print(f"Shape de rendimientos logarítmicos: {returns.shape}")

### 1. Inferencia Topológica (Correlación -> Distancia)
La métrica fundamental para el ML no supervisado en finanzas transforma la correlación cruzada en una matriz de distancias métricas verificables, cumpliendo la desigualdad del triángulo.

In [ ]:
cov_matrix = returns.cov()
corr_matrix = returns.corr()

# Metric Distance: D_ij = sqrt(0.5 * (1 - rho_ij))
dist_matrix = np.sqrt(0.5 * (1 - corr_matrix).round(10)) # precision round

### 2. Agrupamiento Jerárquico (Ward's Linkage)
Aplicamos el algoritmo Linkage sobre la matriz de distancias para descubrir la estructura arborescente (topología) inherente a los retornos de las acciones.

In [ ]:
# Convert metrics distance to condensed distance list for Scipy linkage
condensed_dist = squareform(dist_matrix, checks=False)

# Hierarchical Clustering algorithm
link_matrix = linkage(condensed_dist, method='ward')

fig, ax = plt.subplots(figsize=(14, 6))
dendrogram(link_matrix, labels=df.columns, ax=ax, leaf_rotation=90)
ax.set_title("Estructura Topológica de Activos (Dendrograma HRP)", fontsize=14)
ax.set_ylabel("Distancia Ward")
plt.show()

### 3. Quasi-Diagonalización y Asignación de Riesgo Recursiva (Recursive Bisection)
A diferencia de Markowitz, que sufre por la inestabilidad en la inversión de matrices singulares $w = \Sigma^{-1}\mu$, HRP recorre recursivamente el grafo resultante asignando capital (inversa de la varianza clúster-por-clúster).

In [ ]:
def get_quasi_diag(link):
    return leaves_list(link)

def get_cluster_var(cov, items):
    cov_ = cov.iloc[items, items]
    ivp = 1.0 / np.diag(cov_) # Inverse Variance Portfolio within cluster
    ivp /= ivp.sum()
    # Variance of the cluster
    w_ = ivp.reshape(-1, 1)
    cluster_var = np.dot(np.dot(w_.T, cov_), w_)[0, 0]
    return cluster_var

def get_hrp_weights(cov, sort_ix):
    w = pd.Series(1.0, index=sort_ix)
    c_items = [sort_ix] # Initialize with all sorted items
    
    while len(c_items) > 0:
        # Bisect
        c_items = [i[int(j):int(k)] 
                   for i in c_items 
                   for j, k in ((0, len(i)/2), (len(i)/2, len(i))) if len(i) > 1]
        
        for i in range(0, len(c_items), 2):
            c_items0 = c_items[i]    # Left cluster
            c_items1 = c_items[i+1]  # Right cluster
            
            c_var0 = get_cluster_var(cov, c_items0)
            c_var1 = get_cluster_var(cov, c_items1)
            
            alpha = 1 - c_var0 / (c_var0 + c_var1)
            
            w[c_items0] *= alpha # Weight for left cluster
            w[c_items1] *= (1 - alpha) # Weight for right cluster
            
    return w

ordered_indices = get_quasi_diag(link_matrix)
hrp_weights = get_hrp_weights(cov_matrix, ordered_indices)

# Reorder back to original asset names to print results
hrp_weights.index = returns.columns[hrp_weights.index]
hrp_weights.sort_values(ascending=False, inplace=True)

print("Top 10 HRP Capital Allocation Weights:")
print(hrp_weights.head(10).round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
sns.barplot(x=hrp_weights.index, y=hrp_weights.values, ax=ax, color="#4C72B0")
ax.set_title("Asignación Vectorial de Capital (Hierarchical Risk Parity)", fontsize=14)
ax.set_ylabel("Ponderación (Weight)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=8)
plt.tight_layout()
plt.show()